In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("okienka").getOrCreate()
spark

In [3]:
czujnik_temperatury = ((12.5, "2019-01-02 12:00:00"),
(17.6, "2019-01-02 12:00:20"),
(14.6,  "2019-01-02 12:00:30"),
(22.9,  "2019-01-02 12:01:15"),
(17.4,  "2019-01-02 12:01:30"),
(25.8,  "2019-01-02 12:03:25"),
(27.1,  "2019-01-02 12:02:40"),
)

In [6]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("temperatura", DoubleType(), True), #na koncu informacja czy pozwalamy zeby wartosc byla pusta czy nie
    StructField("czas", StringType(), True),
])

SCHEMA = """
temperatura DOUBLE, czas STRING
""" # DDL # """ to jakiś długi string

In [7]:
df = (spark.createDataFrame(czujnik_temperatury, schema=schema)
      .withColumn("czas", to_timestamp("czas")))

df2 = (spark.createDataFrame(czujnik_temperatury, schema=SCHEMA)
      .withColumn("czas", to_timestamp("czas")))

In [8]:
df.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [9]:
df2.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [10]:
df.show(2)

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
+-----------+-------------------+
only showing top 2 rows



In [11]:
df2.show(2)

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
+-----------+-------------------+
only showing top 2 rows



In [12]:
df.createOrReplaceTempView("df") #widok przetrzymuje kod generujący te dane
#pracując w sparku musimy zrobić widok

In [13]:
spark.sql("select czas, temperatura from df where temperatura > 21").show(5)

+-------------------+-----------+
|               czas|temperatura|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [14]:
# Thumbling window

import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","30 seconds")).count()
df2.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:30}|2    |
|{2019-01-02 12:00:30, 2019-01-02 12:01:00}|1    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:30}|1    |
|{2019-01-02 12:01:30, 2019-01-02 12:02:00}|1    |
|{2019-01-02 12:03:00, 2019-01-02 12:03:30}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+



In [15]:
df2.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- count: long (nullable = false)



In [ ]:
#krotkie powtorzenie to bylo

In [ ]:
#teraz strumieniowanie strukturalne

#kazdy strumien bedzie generowany za pomocą 3 rzeczy: 1.źródło, 
#2. transformacje danych, które chcemy wykonać (możemy też nie zmieniać niczego), 
#3. miejsce gdzie "wypływa", do plików, do Kafki jako topic - u nas w Terminalu konsoli

In [ ]:
# domyslnie spark nie wspiera połączenia do Kafki, potrzebujemy konektor (biblioteka)
# biblioteka w sparku to są pliki z rozszerzeniem jar(?)

In [1]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)
# nie ma transformacji 
# ukryta transformacja - df_transform = df lub df = 1* df

query = (df.writeStream 
    .format("console") 
    #.outputMode("append") - to jest automatycznie ustawione na append dlatego nie musimy pisać
    .option("truncate", False) 
    .start()
) 

query.awaitTermination() #jak piszemy skrypt w notatniku to musimy to dać na końcu

Writing streamrate.py


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

#moge ten sam kod robic w notatniku ale musze miec ta funckje procesujaca
def process_batch(df, batch_id, tstop=5): #zmienna sztuczna tstop zeby przestac 
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch) #dla kazdego batcha wykonujemy funkcje process_batch, 
        # ta funkcja wypisze nam na ekranie oprocz tego ze wypisuje sie w terminalu, wypisze tez numer batcha
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Batch ID: 1
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:39:28.319|0    |
|2026-04-20 10:39:29.319|1    |
+-----------------------+-----+

Batch ID: 2
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:39:30.319|2    |
|2026-04-20 10:39:31.319|3    |
+-----------------------+-----+

Batch ID: 3
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:39:32.319|4    |
+-----------------------+-----+

Batch ID: 4
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:39:33.319|5    |
+-----------------------+-----+

Batch ID: 5
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:39:34.319|6    |
+------------------

In [2]:
spark.stop() #musimy zatrzymac sparka, albo zrestartowac kernel

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10")) #expr - experience - za pomoca napisu stringowego generuje jakas wartosc
        .select("czas", "temperatura")
       )

query = (stream.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 1
+-----------------------+------------------+
|czas                   |temperatura       |
+-----------------------+------------------+
|2026-04-20 10:38:47.888|24.24966785257629 |
|2026-04-20 10:38:48.888|29.248025302211417|
+-----------------------+------------------+

Batch ID: 2
+-----------------------+------------------+
|czas                   |temperatura       |
+-----------------------+------------------+
|2026-04-20 10:38:49.888|27.06473157194048 |
|2026-04-20 10:38:50.888|29.128523211400957|
+-----------------------+------------------+

Batch ID: 3
+-----------------------+------------------+
|czas                   |temperatura       |
+-----------------------+------------------+
|2026-04-20 10:38:51.888|23.786745644940552|
+-----------------------+------------------+

Batch ID: 4
+-----------------------+-----------------+
|czas                   |temperatura      |
+------

In [1]:
#strumieniowo filtrować
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura") > 25)

query = (stream_filtered.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 1
+----------------------+------------------+
|czas                  |temperatura       |
+----------------------+------------------+
|2026-04-20 10:43:36.22|25.885833237893486|
+----------------------+------------------+

Batch ID: 2
+----------------------+------------------+
|czas                  |temperatura       |
+----------------------+------------------+
|2026-04-20 10:43:39.22|25.524792799624283|
+----------------------+------------------+

Batch ID: 3
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 4
+----------------------+------------------+
|czas                  |temperatura       |
+----------------------+------------------+
|2026-04-20 10:43:41.22|26.834927554069203|
+----------------------+------------------+

Batch ID: 5
+----------------------+------------------+
|czas                  |temperatura       |
+----------------------

In [1]:
#wykrywanie anomali
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_anomaly = stream.withColumn(
    "anomaly",
    when(col("temperatura") > 29.5, "TAK").otherwise("NIE")
)

query = (stream_anomaly.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+-------+
|czas|temperatura|anomaly|
+----+-----------+-------+
+----+-----------+-------+

Batch ID: 1
+-----------------------+------------------+-------+
|czas                   |temperatura       |anomaly|
+-----------------------+------------------+-------+
|2026-04-20 10:44:57.446|29.81729193178584 |TAK    |
|2026-04-20 10:44:58.446|28.558482541154355|NIE    |
+-----------------------+------------------+-------+

Batch ID: 2
+-----------------------+-----------------+-------+
|czas                   |temperatura      |anomaly|
+-----------------------+-----------------+-------+
|2026-04-20 10:44:59.446|28.55585039161967|NIE    |
+-----------------------+-----------------+-------+

Batch ID: 3
+-----------------------+------------------+-------+
|czas                   |temperatura       |anomaly|
+-----------------------+------------------+-------+
|2026-04-20 10:45:00.446|26.436077018241996|NIE    |
+-----------------------+------------------+-------

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window
from pyspark.sql.functions import col, expr

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))

tumbling_window = (stream
    .groupBy(window(col("czas"), "10 seconds"))
    .avg("temperatura").alias("srednia")
)

query = (tumbling_window.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+------+----------------+
|window|avg(temperatura)|
+------+----------------+
+------+----------------+

Batch ID: 1
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 10:50:40, 2026-04-20 10:50:50}|22.806690566445543|
|{2026-04-20 10:50:30, 2026-04-20 10:50:40}|23.49421063730999 |
+------------------------------------------+------------------+

Batch ID: 2
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 10:50:40, 2026-04-20 10:50:50}|23.262820236368036|
|{2026-04-20 10:50:50, 2026-04-20 10:51:00}|24.350420514018612|
|{2026-04-20 10:50:30, 2026-04-20 10:50:40}|23.49421063730999 |
+------------------------------------------+------------------+

Batch ID: 3
+------------

In [2]:
#sliding window
from pyspark.sql import SparkSession
from pyspark.sql.functions import window
from pyspark.sql.functions import col, expr

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))


sliding = (stream.groupBy(window(col("czas"), "10 seconds", "5 seconds"))
           .avg("temperatura")
          )

query = (sliding.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+------+----------------+
|window|avg(temperatura)|
+------+----------------+
+------+----------------+

Batch ID: 1
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 10:52:30, 2026-04-20 10:52:40}|27.929536990767946|
|{2026-04-20 10:52:40, 2026-04-20 10:52:50}|24.71415956170451 |
|{2026-04-20 10:52:45, 2026-04-20 10:52:55}|25.243881342574834|
|{2026-04-20 10:52:35, 2026-04-20 10:52:45}|24.80862098248981 |
+------------------------------------------+------------------+

Batch ID: 2
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 10:52:50, 2026-04-20 10:53:00}|26.873801748622686|
|{2026-04-20 10:52:30, 2026-04-20 10:52:40}|27.929536990767946|
|{2026-04-20 10:52:55, 202

In [3]:
%%file generator.py
# generator.py
import json, os, random, time
from datetime import datetime, timedelta

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

# Simulate file-based streaming
while True:
    batch = [generate_transaction() for _ in range(10)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)



Writing generator.py


In [1]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, to_timestamp
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("jsonDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=20):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
    


stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

query = (stream.writeStream
         .format("console")
         .foreachBatch(process_batch)
         .start())

Batch ID: 0
+------+-------+-------+--------+-----------+--------------------------+
|tx_id |user_id|amount |store   |category   |timestamp                 |
+------+-------+-------+--------+-----------+--------------------------+
|TX3460|u20    |1355.33|Gdańsk  |książki    |2026-04-20T11:02:23.666338|
|TX4541|u20    |4259.34|Kraków  |żywność    |2026-04-20T11:02:23.666378|
|TX9720|u16    |4643.25|Wrocław |książki    |2026-04-20T11:02:23.666390|
|TX4524|u01    |3763.55|Gdańsk  |elektronika|2026-04-20T11:02:23.666407|
|TX2566|u15    |780.82 |Gdańsk  |książki    |2026-04-20T11:02:23.666415|
|TX5883|u02    |3664.21|Wrocław |książki    |2026-04-20T11:02:23.666421|
|TX4167|u04    |1923.76|Wrocław |odzież     |2026-04-20T11:02:23.666427|
|TX4919|u09    |4166.24|Kraków  |żywność    |2026-04-20T11:02:23.666438|
|TX6946|u16    |1530.66|Gdańsk  |książki    |2026-04-20T11:02:23.666445|
|TX5073|u01    |3554.14|Wrocław |elektronika|2026-04-20T11:02:23.666463|
|TX2890|u06    |1334.55|Gdańsk  |żywnoś

In [ ]:
#sprobowac zrobic jakies filtrowanie albo agregat po sklepach
#przychodzi nowy plik to ile bylo transakcji per sklep, albo srednia suma